# MuQ: sumber FP32 dari bobot training

Tahap pertama sebelum kompresi MuQ: ekspor hidden-state 10 dari snapshot Hugging Face yang sama dengan cache training. Tidak mengubah MusicFM palette6, head palette6, anotasi, atau model aplikasi.

Jalankan dari root Section-Annotations dengan kernel `.venv-songformer311`. Membutuhkan macOS dan paket Core ML lokal. Sediakan beberapa GiB disk dan tutup proses training/inferensi lain. Ekspor ini belum mengukur F1 seluruh validation atau kecepatan iPhone.

In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, sys, json
assert Path('finetuning/config.json').is_file(), 'Buka notebook dari root Section-Annotations'
SONGMAP_ROOT = Path('/Users/danendra/Documents/ADA26/C6/SongMap')
assert (SONGMAP_ROOT/'SongMap/Resources/MLModels/SongFormer/songformer_coreml_manifest.json').is_file()
output = Path('outputs.local') / ('muq-fp32-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
print('Python:', sys.executable)
print('Output:', output)


In [ ]:
log = output.with_suffix('.log')
log.parent.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'deployment.export_muq_fp32',
           '--songmap-root', str(SONGMAP_ROOT), '--output-dir', str(output)]
with log.open('w', encoding='utf-8') as handle:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            handle.write(line)
            handle.flush()
        status = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
if status:
    raise RuntimeError(f'Ekspor gagal ({status}); log: {log.resolve()}')


In [ ]:
report = json.loads((output/'export_report.json').read_text())
for key in ('muq_revision', 'source_kind', 'wrapper_vs_native_encoder',
            'frontend_conv_app_vs_original', 'combined_frontend_rotary_effect',
            'trace_long_window', 'checks', 'source_window_passed'):
    print(key + ':', json.dumps(report.get(key), indent=2))
print('Report:', (output/'export_report.json').resolve())


## Setelah selesai

Bagikan `export_report.json`. `source_window_passed` memeriksa konversi encoder pada input nyata (prefix dan satu window panjang), bukan persetujuan integrasi. Perbedaan frontend/rotary dilaporkan terpisah dan tidak disembunyikan. Langkah berikutnya adalah full validation dengan MusicFM palette6 dan head palette6 tetap, lalu INT8/palette6 MuQ dibandingkan terhadap sumber FP32 ini. Jangan mengganti aset SongMap dulu.